# Transformers

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import lightning as L
from torch.utils.data import TensorDataset, DataLoader

## Create the Datasets that we will use.

We will translate **Let's go** and **to go** into spanish. **Let's go** should translate to **vamos<EOS>** and **to go** should translate to **ir<EOS>**

In [3]:
input_vocab = {'<SOS>': 0,
               'lets': 1,
               'to': 2,
               'go': 3}

output_vocab = {'<SOS>': 0,
                'ir': 1,
                'vamos': 2,
                'y': 3,
                '<EOS>':4}

inputs = torch.tensor([[1, 3],
                       [2, 3]])

labels = torch.tensor([[2],
                       [1]])

dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

## Position Encoding

Helps the transformer keep track of the order of the words in the input and the output.
To do that we use a series of alternating sine and cosine curves.
Since the position encoding values never change (the first token always uses the same position encoding values regardless of what that token is) we can precompute them and save them in a lookup table (that makes adding position encoding values super fast). 

**The PositionEncoding class takes word embeddings and adds a unique positional signal to each one, so the transformer knows the order of the tokens in the sequence.**

In [4]:
class PositionEncoding(nn.Module):
    def __init__(self, d_model=2, max_len=3):
        # d_model: The dimension of the transformer (also the number of embedding values per token)
        #           In 'Attention Is All You Need' d_model was set as d_model=512
        # max_len: maximum number of tokens we allow as input. We will use only 2 word phrases + <SOS>, so we set max_len=3

        super().__init__()

        # We create the lookup table of position encoding values and initialize all of them to 0
        # To do that we create a matric of 0s that has max_len rows and d_model columns. 
        # A lookup table is just a pre-computed table where you say "give me the row for position 2" and you get back [..., ...]. 
        # No calculation needed at that point — it was all computed once upfront when you built the table.
        pe = torch.zeros(max_len, d_model)

        # We create a sequence of numbers for each position that a token can have in the input or output. 
        # We change them to floats and we use .unsqueeze(1) to convert a list of numbers into a matrix with one row for each index and all indices in a single column.
        # In this example with max_len = 3 we create a matrix with 3 rows and 1 column.
        position = torch.arange(start=0, end=max_len, step=1).float().unsqueeze(1) # This gives us [[0], [1], [2]]  (a column vector of shape (3, 1).)


        # Now we determine the y-axis coordinates on the sine and cosine curves
        div_term = 1/torch.tensor(10000.0)**(torch.arange(start=0, end=d_model, step=2).float()/d_model)

        # We calculate the actual positional encoding values (pe was initialized as a tensor of 0s)
        pe[:, 0::2] = torch.sin(position * div_term) # Every other column, starting with the 1st, has sin() values
        pe[:, 1::2] = torch.cos(position * div_term) # Every other column, starting with the 2nd, has cos() values
        # In the code above, before the ',' is the row and by using ':' we select all rows. After the ',' is the column
        # and by using 0::2 we say 'start from column 0 and go to the end with a stepsize of 2



        # Next we register pe. 
        # A PyTorch model as having two kinds of numbers stored inside it:
        # 1) Trainable parameters — weights and biases that the optimizer adjusts during training (e.g. attention weights). These live in model.parameters().
        # 2) Buffers — fixed numbers the model needs but should never change during training.
        # pe belongs in the second category. The positional encoding values are pure math — sine and cosine — not something the model should be tweaking. 
        # So we use register_buffer to say:
        # "Store this inside the model, but keep it away from the optimizer."
        self.register_buffer('pe', pe) # Using 'register_buffer()' prevents 'pe' from getting passed to the optimizer

    # The forward method is what is called by default when we use a PositionEncoding() object. 
    # When we create a PositionEncoding() object, "pe=PositionEncoding()", we can pass position encoding values to the word embeddings with pe(word_embeddings)
    def forward(self, x):
        # x = word embedding values
        # x.size(0) returns the number of tokens in the input sequence
        return x + self.pe[:x.size(0), :] 

## Attention

We will code the 'Attention' class to do all of the types of attention that a transformer might need: **Self-Attention, Masked Self-Attention** (this one is used by the Decoder during training) and **Encoder-Decoder Attention**

**Self Attention** is the type of attention used in Encoder-Decoder transformers and Encoder-Only Transformers and allows every word in a phrase to define a relationship with any other word in the phrase (regardless of the order of the words). 

**Masked Self-Attention** is used by all types of transformers and it allows each word in a phrase to define a relationship with itself and the words that came before it.

**Encoder-Decoder Attention** is only used in Encoder-Decoder transformers, where there is a distinct seperation of the part of the transformer that processes in the input (the encoder) from the part that generates the output (the decoder). Encoder-Decoder Attention lets each word in the output (the decoder) to define relationships with all the words in the input (in the encoder)

**The Attention class takes input representations, projects them into queries/keys/values, computes how much each token should attend to every other token, and returns a new representation for each token that blends information from the whole sequence.**

In [ ]:
class Attention(nn.Module):
    def __init__(self, d_model=2):
        super().__init__()
        # We initialize the Weights we will use to create the query (q), key (k), and value (v) for each token
        # Most implementations include bias terms but we dont include them as they werent part of the original Attention is All You Need paper
        self.W_q = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_k = nn.Linear(in_features=d_model, out_features=d_model, bias=False)
        self.W_v = nn.Linear(in_features=d_model, out_features=d_model, bias=False)

        # We set those here for simplicity down the road
        self.row_dim = 0 # row dimension is 0
        self.col_dim = 1 # column dimension is 1
    
    def forward(self, encodings_for_q, encodings_for_k, encodings_for_v, mask=None):
        # We create the Query, Key, and Values using the encodings associated with each token 
        # For normal Self-Attention and Masked Self-Attention encodings_for_q == encodings_for_k == encodings_for_v
        # For Encoder-Decoder Attention, encodings_for_q comes from the Decoder and encodings_for_k and encodings_for_v are different and come from the Encoder
        q = self.W_q(encodings_for_q)
        k = self.W_k(encodings_for_k)
        v = self.W_v(encodings_for_v)

        # We compute the raw attention scores using the equation (q * K^T) / sqrt(d_model)
        sims = torch.matmul(q, k.transpose(dim0=self.row_dim, dim1=self.col_dim))
        scaled_sims = sims / torch.tensor(q.size(self.col_dim)**0.5)

        if mask is not None:
            # We replace things we want masked out with a bery large negative number (to approximate -infinity) so that
            # Softmax() will give all masked elements an output value of 0
            '''
            `scaled_sims` is a `(seq_len, seq_len)` matrix where each cell `[i, j]` means "how much should token i attend to token j?"
            For a 3-token sequence it looks like this:

                        <SOS>   cat   sat
            <SOS>        [ 0.9,  0.3,  0.8 ]   ← how much <SOS> attends to each token
            cat          [ 0.5,  0.7,  0.2 ]   ← how much "cat" attends to each token
            sat          [ 0.1,  0.4,  0.6 ]   ← how much "sat" attends to each token
           

            In the decoder, token `i` is not allowed to look at tokens that come **after** it — that would be peeking at future words the model hasn't generated yet. So we block out the upper triangle:
       
                        <SOS>   cat   sat
            <SOS>        [ 0.9,  -1e9, -1e9 ]   ← <SOS> can only see itself
            cat          [ 0.5,  0.7,  -1e9 ]   ← "cat" can see <SOS> and itself
            sat          [ 0.1,  0.4,  0.6  ]   ← "sat" can see everything
            When softmax runs on a row, -1e9 becomes essentially 0 — those positions are completely ignored. That's the mask.
            '''
            scaled_sims = scaled_sims.masked_fill(mask=mask, value=-1e9)

        # Next we apply Softmax to determine what percent of each token's value to use in the final attention values
        attention_percents = F.softmax(scaled_sims, dim=self.col_dim) # Softmax converts the raw scores into percentages that sum to 1

        # Scale the values by their associated percentages and add them up
        attention_scores = torch.matmul(attention_percents, v)
        '''
        After softmax, `attention_percents` looks something like this (each row sums to 1):

                    <SOS>   cat   sat
        <SOS>        [ 1.0,   0.0,  0.0 ]
        cat          [ 0.4,   0.6,  0.0 ]
        sat          [ 0.1,   0.3,  0.6 ]


        And `v` (the value matrix) looks like:

                dim0   dim1
        <SOS>  [ 0.2,   0.5 ]
        cat    [ 0.9,   0.1 ]
        sat    [ 0.4,   0.8 ]


        The matrix multiplication blends the value vectors using those percentages. For "cat" for example:

        cat output = 0.4 × [0.2, 0.5]   (40% of <SOS>'s value)
                + 0.6 × [0.9, 0.1]   (60% of cat's own value)
        
        each token's output is a blend of information from the tokens it attended to, weighted by how much attention it paid to each one.
        That blended vector is what gets passed forward in the network.

        '''

        return attention_scores

